# MNPS — Zero‑Shot Classification + Ground Truth Refinement (No Few‑Shot Exemplars in Prompt)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Main-Version-2/MNPS_ZeroShot_GT_Refine.ipynb)

**Source Files**
- `Batch Input.csv` *(to classify)*
- `MNPS Roles.csv` and `MNPS KSACs.csv` *(definitive role set + KSACs)*
- `Ground Truth Masterfile.csv` *(89 labeled examples used for **post‑classification refinement** and evaluation)*
- Optional nuance: `Competency Extended Descriptions.csv`, `Korn_Ferry Lominger 38 Competencies.csv`

**Minor Sub‑Group Policy**: Allowed = **Lead, I, II, III**. If the model proposes **IV/4**, treat as **Lead** **iff** leadership/escalation/mentorship signals are present in the job text; otherwise map to **III**.

### Zero‑Shot Prompt (Section 4)

```
Objective: Evaluate and group jobs from the "Batch Input.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you, including "MNPS Roles.csv" and "MNPS KSACs.csv".
- Group jobs based on similarities into:
  - Major role groupings from the comprehensive list provided in "MNPS Roles.csv" (e.g., Specialist, Analyst, Manager, Technician, Para Pro, Advisor, etc.)
  - Minor sub-groupings (e.g., Specialist I, II, III, IV)
- **Crucially, use the "MNPS Roles.csv" and "MNPS KSACs.csv" documents as the definitive and comprehensive lists of valid Major Role Groups and their corresponding Knowledge, Skills, Abilities, and Competencies (KSACs) to guide your classification.**
- Use the remaining documents ("Competency Extended Descriptions.csv" and "Korn_Ferry Lominger 38 Competencies.csv") to help you clarify subtle differences in role groupings and sub-groupings and to inform the grouping justification.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III"). Ensure the Role comes from the list in "MNPS Roles.csv".

Additional Guidelines:

- Ensure all sources used are cited properly in the justification.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks, and explicitly referencing the relevant KSACs and roles from the provided documents.
- Run in batch over "Batch Input.csv" with conservative settings (temperature=0.2) for reproducibility.
```

**MNPS Minor Sub‑Group Clarification:** Approved minor levels are **Lead, I, II, III**. Interpret any model‑proposed **IV/4** as **Lead** *if* the KSACs/Functions clearly indicate leadership/escalation/mentorship; otherwise map to **III**.

In [ ]:
!pip -q install scikit-learn openai pandas

import os, re, json, time
from typing import Dict, List, Tuple
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --------------------------
# CONFIG
# --------------------------
BATCH_INPUT_CSV  = "Batch Input.csv"
ROLES_CSV        = "MNPS Roles.csv"
KSACS_CSV        = "MNPS KSACs.csv"
GT_CSV           = "Ground Truth Masterfile.csv"
COMP_EXT_CSV     = "Competency Extended Descriptions.csv"    # optional
LOMINGER_CSV     = "Korn_Ferry Lominger 38 Competencies.csv" # optional

OUTPUT_PRED_CSV  = "classified_job_descriptions_pre_refine.csv"
OUTPUT_FINAL_CSV = "classified_job_descriptions_final.csv"
DECISION_LOG_CSV = "classification_decision_log.csv"

# LLM settings (wire your provider)
MODEL_PROVIDER   = "openai"
MODEL_NAME       = "gpt-4o-mini"
TEMPERATURE      = 0.2  # per prompt
TOP_K_CANDIDATES = 10   # retrieval narrowing


In [ ]:
def read_csv_robust(path: str):
    # Try common encodings
    for enc in ["utf-8", "latin1", "windows-1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    # final attempt
    return pd.read_csv(path, encoding="latin1", errors="ignore")

def norm_col(s: str) -> str:
    return re.sub(r'[^a-z0-9]+','_',str(s).strip().lower())

APPROVED_MINOR = {"Lead","I","II","III"}

def looks_like_lead(fields: Dict[str,str]) -> bool:
    text = " ".join([
        str(fields.get("Essential Functions","")),
        str(fields.get("Knowledge, Skills and Abilities","")),
        str(fields.get("Position Summary",""))
    ]).lower()
    lead_signals = [
        "lead", "leads", "team lead", "mentors", "mentorship",
        "coaches others", "escalation", "tier 3", "senior-most",
        "principal", "subject matter expert", "sme",
        "oversees", "coordinates others", "assigns work", "guides staff"
    ]
    return any(sig in text for sig in lead_signals)

def normalize_minor(sub: str, fields: Dict[str,str]) -> str:
    s = "" if sub is None else str(sub).strip()
    if s.lower() == "lead":
        return "Lead"
    if s in {"I","II","III"}:
        return s
    m = re.search(r"\b(I|II|III|IV)\b", s.upper())
    if m:
        roman = m.group(1)
        if roman in {"I","II","III"}:
            return roman
        if roman == "IV":
            return "Lead" if looks_like_lead(fields) else "III"
    if s.isdigit():
        if s in {"1","2","3"}:
            return {"1":"I","2":"II","3":"III"}[s]
        if s == "4":
            return "Lead" if looks_like_lead(fields) else "III"
    # texty variants
    m2 = re.search(r"\b(level|tier)\s*(1|2|3|4|i|ii|iii|iv)\b", s.lower())
    if m2:
        val = m2.group(2).upper()
        if val in {"I","II","III"}:
            return val
        if val in {"IV","4"}:
            return "Lead" if looks_like_lead(fields) else "III"
    # fallback: choose III unless explicit lead signals
    return "Lead" if looks_like_lead(fields) else "III"


In [ ]:
roles_df = read_csv_robust(ROLES_CSV)
ksacs_df = read_csv_robust(KSACS_CSV)
gt_df    = read_csv_robust(GT_CSV)

roles_df.columns = [norm_col(c) for c in roles_df.columns]
ksacs_df.columns = [norm_col(c) for c in ksacs_df.columns]
gt_df.columns    = [norm_col(c) for c in gt_df.columns]

# Identify key columns
role_col = [c for c in roles_df.columns if "role" in c][-1]
ks_role  = [c for c in ksacs_df.columns if "role" in c][-1]
ks_text  = [c for c in ksacs_df.columns if "ksac" in c or "knowledge" in c][-1]

# Build closed set of roles from MNPS Roles.csv
VALID_ROLES = set(roles_df[role_col].astype(str).str.strip())

# Build role -> KSAC text corpus
role_ksac = (
    ksacs_df.groupby(ks_role)[ks_text]
            .apply(lambda s: " ".join(map(lambda x: "" if pd.isna(x) else str(x), s)))
            .to_dict()
)

# GT columns for refinement & eval
gt_name = [c for c in gt_df.columns if "job" in c and "name" in c]
gt_major = [c for c in gt_df.columns if "major" in c and "role" in c]
gt_minor = [c for c in gt_df.columns if "minor" in c]
gt_just  = [c for c in gt_df.columns if "justif" in c]

GT_COLS = {
    "name":  gt_name[0]  if gt_name  else None,
    "major": gt_major[0] if gt_major else None,
    "minor": gt_minor[0] if gt_minor else None,
    "just":  gt_just[0]  if gt_just  else None,
}

print("Closed role set size:", len(VALID_ROLES))


In [ ]:
# Build TF-IDF over role KSAC corpus
role_names = list(role_ksac.keys())
role_texts = [role_ksac.get(r, "") for r in role_names]
vec = TfidfVectorizer(min_df=1, max_df=0.9, ngram_range=(1,2))
X_roles = vec.fit_transform(role_texts)

def topK_roles(desc_blob: str, K: int = 10) -> List[str]:
    q = vec.transform([desc_blob])
    sims = cosine_similarity(q, X_roles).ravel()
    idx = sims.argsort()[::-1][:K]
    return [role_names[i] for i in idx]


In [ ]:
PROMPT_TMPL = """    Objective: Evaluate and group this MNPS job based on similarities in job functions, not job titles.

You have access to MNPS Roles (closed label set) and KSACs. Choose only from the provided candidate roles.
Approved minor sub-groups: "Lead", "I", "II", "III". If "IV/4" is inferred, interpret as "Lead" only if KSACs/Functions show leadership/escalation/mentorship; otherwise "III".

Output STRICT JSON with keys: major_role_group, minor_sub_group, new_job_title, grouping_justification.
Cite sources in justification (e.g., "MNPS KSACs: Technician", "MNPS Roles").

Candidate Roles: {candidates}

Position Summary: {summary}
Essential Functions: {functions}
Education: {education}
Work Experience: {experience}
Licenses/Certifications: {licenses}
Knowledge, Skills and Abilities: {ksac}
"""

def build_prompt(row: pd.Series) -> Tuple[str, Dict[str,str]]:
    fields = {
        "Position Summary": row.get("Position Summary",""),
        "Essential Functions": row.get("Essential Functions",""),
        "Education": row.get("Education",""),
        "Work Experience": row.get("Work Experience",""),
        "Licenses/Certifications": row.get("Licenses and Certifications",""),
        "Knowledge, Skills and Abilities": row.get("Knowledge, Skills and Abilities",""),
    }
    blob = " ".join(str(v) for v in fields.values())
    candidates = topK_roles(blob, K=TOP_K_CANDIDATES)
    prompt = PROMPT_TMPL.format(
        candidates=", ".join(candidates),
        summary=fields["Position Summary"][:2000],
        functions=fields["Essential Functions"][:2000],
        education=fields["Education"][:1200],
        experience=fields["Work Experience"][:1200],
        licenses=fields["Licenses/Certifications"][:800],
        ksac=fields["Knowledge, Skills and Abilities"][:1200],
    )
    return prompt, fields


In [ ]:
def call_llm_json(prompt: str) -> str:
    # Implement your provider call. Example: OpenAI JSON mode.
    import os
    if MODEL_PROVIDER == "openai":
        from openai import OpenAI
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY not set in environment.")
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=TEMPERATURE,
            response_format={"type": "json_object"},
            messages=[
                {"role":"system","content":"You are a careful classifier that returns strict JSON."},
                {"role":"user",  "content": prompt},
            ]
        )
        return resp.choices[0].message.content
    else:
        raise NotImplementedError("MODEL_PROVIDER not supported here.")


In [ ]:
in_df = read_csv_robust(BATCH_INPUT_CSV)
required = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
miss = [c for c in required if c not in in_df.columns]
if miss:
    raise ValueError(f"Missing required columns in Batch Input: {miss}")

work_df = in_df.reset_index().rename(columns={"index":"row_id"}).copy()

preds = []
logs  = []
for _, row in work_df.iterrows():
    prompt, fields = build_prompt(row)
    raw = call_llm_json(prompt)   # returns strict JSON string
    rec = json.loads(raw)

    # Hard validation
    role = (rec.get("major_role_group","") or "").strip()
    if role not in VALID_ROLES:
        raise ValueError(f"Predicted role '{role}' not in MNPS Roles.csv")

    minor = normalize_minor(rec.get("minor_sub_group",""), fields)
    rec["minor_sub_group"] = minor

    # Keep prompt & raw for traceability
    logs.append({
        "row_id": row["row_id"],
        "job_name": row["Job Description Name"],
        "prompt_preview": prompt[:1000],
        "raw_response": raw
    })
    preds.append(rec)

pred_df = pd.DataFrame(preds)
out_pre = pd.concat([work_df[["row_id","Job Description Name"]], pred_df], axis=1)
out_pre["original_job_title"] = out_pre["Job Description Name"]
out_pre = out_pre[[
    "original_job_title","new_job_title","major_role_group","minor_sub_group","grouping_justification",
    "Job Description Name","row_id"
]]

out_pre.to_csv(OUTPUT_PRED_CSV, index=False, encoding="utf-8")
pd.DataFrame(logs).to_csv(DECISION_LOG_CSV, index=False, encoding="utf-8")
print("Saved pre-refinement predictions ->", OUTPUT_PRED_CSV)


In [ ]:
# Build TF-IDF over GT justification + KSAC text for similarity
def build_gt_corpus(gt_df: pd.DataFrame) -> Tuple[TfidfVectorizer, pd.DataFrame, any]:
    text_cols = []
    if GT_COLS["just"] and GT_COLS["just"] in gt_df.columns:
        text_cols.append(GT_COLS["just"])
    # Join with KSACs by role to augment
    join = gt_df.copy()
    if GT_COLS["major"]:
        ks_map = {k: role_ksac.get(k,"") for k in VALID_ROLES}
        join["__ksac_blob__"] = join[GT_COLS["major"]].map(ks_map)
        text_cols.append("__ksac_blob__")
    join["__text__"] = join[text_cols].apply(lambda r: " ".join(map(str, r.values)), axis=1)
    vec = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1,2))
    X = vec.fit_transform(join["__text__"])
    return vec, join, X

vec_gt, gt_join, X_gt = build_gt_corpus(gt_df)

def refine_one(row: pd.Series) -> Tuple[str,str,str,str,str]:
    """Return (final_major, final_minor, final_title, final_just, reason_log)"""
    # Compose query text from the model's own justification + the job's determinant fields
    q_text = " ".join([
        str(row.get("grouping_justification","")),
        str(row.get("Position Summary","")),
        str(row.get("Essential Functions","")),
        str(row.get("Knowledge, Skills and Abilities",""))
    ])
    q = vec_gt.transform([q_text])
    sims = cosine_similarity(q, X_gt).ravel()
    top = sims.argsort()[::-1][:5]  # take a few best GT exemplars

    # Start with model's own prediction
    major_pred = row["major_role_group"]
    minor_pred = row["minor_sub_group"]
    title_pred = row["new_job_title"]
    just_pred  = row["grouping_justification"]
    reason = []

    # If a top GT exemplar strongly suggests a different role, consider switching
    # Thresholds can be tuned; start simple.
    best_idx = top[0]
    best_sim = sims[best_idx]
    gt_row = gt_join.iloc[best_idx]
    gt_role = gt_row[GT_COLS["major"]] if GT_COLS["major"] else ""
    gt_minor = gt_row[GT_COLS["minor"]] if GT_COLS["minor"] else ""
    gt_just  = gt_row[GT_COLS["just"]]  if GT_COLS["just"]  else ""

    # Heuristic: if similarity >= 0.35 and roles differ, switch to GT role
    if best_sim >= 0.35 and gt_role and gt_role in VALID_ROLES and gt_role != major_pred:
        reason.append(f"Role refined from {major_pred} -> {gt_role} based on GT similarity {best_sim:.2f}")
        major_pred = gt_role

    # Adjust minor by GT if GT minor exists
    if gt_minor:
        fields = {
            "Position Summary": row.get("Position Summary",""),
            "Essential Functions": row.get("Essential Functions",""),
            "Knowledge, Skills and Abilities": row.get("Knowledge, Skills and Abilities","")
        }
        minor_pred = normalize_minor(gt_minor, fields)
        reason.append(f"Minor set to {minor_pred} using GT exemplar")

    # Merge justifications: keep model's but add GT citation
    final_just = (str(just_pred) + " | Refined using GT exemplar: " + str(gt_just)).strip()

    return major_pred, minor_pred, title_pred, final_just, "; ".join(reason)

# Join determinant fields to predictions for refinement
enrich = work_df.merge(out_pre[["row_id","major_role_group","minor_sub_group","new_job_title","grouping_justification"]],
                       on="row_id", how="left")

finals = []
logs_refine = []
for _, r in enrich.iterrows():
    major, minor, title, just, why = refine_one(r)
    finals.append({"row_id": r["row_id"], "final_major_role_group": major,
                   "final_minor_sub_group": minor, "final_new_job_title": title,
                   "final_grouping_justification": just})
    logs_refine.append({"row_id": r["row_id"], "job_name": r["Job Description Name"],
                        "refine_reason": why})

final_df = pd.DataFrame(finals)
out_final = (work_df[["row_id","Job Description Name"]]
             .merge(final_df, on="row_id", how="left"))
out_final["original_job_title"] = out_final["Job Description Name"]
out_final = out_final[[
    "original_job_title","final_new_job_title","final_major_role_group","final_minor_sub_group","final_grouping_justification",
    "Job Description Name","row_id"
]].rename(columns={
    "final_new_job_title":"new_job_title",
    "final_major_role_group":"major_role_group",
    "final_minor_sub_group":"minor_sub_group",
    "final_grouping_justification":"grouping_justification"
})

out_final.to_csv(OUTPUT_FINAL_CSV, index=False, encoding="utf-8")
print("Saved final refined predictions ->", OUTPUT_FINAL_CSV)


In [ ]:
if all(GT_COLS.values()):
    gt_eval = gt_df.copy()
    gt_eval["_join_name"] = gt_eval[GT_COLS["name"]].astype(str).str.strip().str.casefold()
    out_eval = out_final.copy()
    out_eval["_join_name"] = out_eval["original_job_title"].astype(str).str.strip().str.casefold()

    joined = out_eval.merge(
        gt_eval[["_join_name", GT_COLS["major"], GT_COLS["minor"]]],
        on="_join_name", how="inner", suffixes=("_pred","_gt")
    )

    print("Overlap with Ground Truth:", len(joined))
    if len(joined):
        acc_major = (joined["major_role_group_pred"] == joined[GT_COLS["major"]]).mean()
        acc_minor = (joined["minor_sub_group_pred"] == joined[GT_COLS["minor"]]).mean()
        print(f"Accuracy — Major: {acc_major:.3f}, Minor: {acc_minor:.3f}")
        display(
            joined.groupby([GT_COLS["major"], "major_role_group_pred"]).size().unstack(fill_value=0)
        )
else:
    print("GT columns not fully detected; skipping evaluation.")


In [ ]:
def generate_narrative_summary(df: pd.DataFrame) -> str:
    """Optionally call LLM to summarize notable patterns and insights."""
    prompt = f"""
    Summarize the notable patterns across MNPS job classifications below.
    Highlight role coverage (e.g., Technician/Advisor/Para Pro vs Analyst/Specialist), minor level distributions (Lead/I/II/III),
    and recurring KSAC themes. Provide 3–6 concise bullets.

    DATA PREVIEW (first 50 rows):
    {df.head(50).to_csv(index=False)}
    """
    # Wire to your LLM if desired; for now, return a simple placeholder.
    return "Narrative summary generation placeholder (wire your LLM to produce a narrative)."

narrative = generate_narrative_summary(out_final)
print(narrative)
